# Session 1 model stability and comparison

This study measures training-seed variability on one fixed, chronological Session 1 split. It does **not** measure variability across data splits. Seeds are `42, 123, 2026, 31415, 8675309`.

All three models use fit attempts from repetitions 1–70, an embargo of repetitions 71–79, and validation attempts from repetition 80 onward. LSTM and GRU use the exact same stride-1 length-10 windows; the window raw-row ID fingerprints are checked against the audited sequence experiment. Each run fits its own scaler and label encoder on fit-partition attempts only. Overlapping windows are formed only within one participant and a contiguous repetition run, after the chronological partition. Fit and validation window rows must be disjoint.

The MLP is trained and evaluated per **single attempt**, while LSTM/GRU predict participant identity from **ten consecutive attempts**. They share the chronological attempt partitions and fit-only preprocessing, but their prediction units, sample counts, and task evidence differ; this is not a perfectly equivalent head-to-head comparison. The existing saved MLP row-holdout results are historical context only and are not loaded here. Session 2 is not accessed.

In [ ]:
from pathlib import Path
import hashlib
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

SEEDS = [42, 123, 2026, 31415, 8675309]
WINDOW_LENGTH = 10
FIT_MAX_REPETITION = 70
VALIDATION_MIN_REPETITION = 80
RECURRENT_UNITS = 64
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001
BATCH_SIZE = 256
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MLP_HIDDEN_LAYER_SIZES = (128, 64)
MLP_ALPHA = 0.001
MLP_LEARNING_RATE = 0.0005
MLP_MAX_ITER = 150
MLP_VALIDATION_FRACTION = 0.15

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'fixed-text.csv'
MODEL_DIR = PROJECT_DIR / 'models'
RESULTS_DIR = PROJECT_DIR / 'results' / 'metrics'
LSTM_VALIDATION_PATH = RESULTS_DIR / 'lstm_repetition_w10_validation.csv'
GRU_VALIDATION_PATH = RESULTS_DIR / 'gru_repetition_w10_validation.csv'
RUNS_OUTPUT_PATH = RESULTS_DIR / 'model_stability_session1_seed_runs_20261022.csv'
SUMMARY_OUTPUT_PATH = RESULTS_DIR / 'model_stability_session1_summary_20261022.csv'
HISTORY_OUTPUT_PATH = RESULTS_DIR / 'model_stability_session1_training_history_20261022.csv'

EXPECTED_FIT_WINDOW_IDS_SHA256 = 'fddb824bb58a983dfa3b4e6eef9905a215e53db82fce0dfa204530c82e599b85'
EXPECTED_VALIDATION_WINDOW_IDS_SHA256 = '2e4d0d0ddae0bd6fecb7f82e37385c75c73b2820f3f172e46489c9d552c5aeb7'

output_paths = [RUNS_OUTPUT_PATH, SUMMARY_OUTPUT_PATH, HISTORY_OUTPUT_PATH]
collisions = [path for path in output_paths if path.exists()]
if collisions:
    raise FileExistsError(f'Refusing to overwrite existing study outputs: {collisions}')
if not LSTM_VALIDATION_PATH.exists() or not GRU_VALIDATION_PATH.exists():
    raise FileNotFoundError('Existing sequence-model validation metrics are required for matched-window comparison.')

try:
    tf.config.experimental.enable_op_determinism()
except RuntimeError as error:
    print('Deterministic-op setting could not be changed after runtime initialization:', error)

print('TensorFlow:', tf.__version__)
print('Seeds:', SEEDS)
print('Project directory:', PROJECT_DIR)

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
df_model = df_raw.copy()
exclusions = [('p020', 2, 4), ('p071', 1, 20), ('p087', 1, 8)]
for participant, session, repetition in exclusions:
    matching = (
        df_model['participant'].eq(participant)
        & df_model['session'].eq(session)
        & df_model['repetition'].eq(repetition)
    )
    if int(matching.sum()) != 1:
        raise ValueError(f'Expected exactly one documented exclusion: {(participant, session, repetition)}')
    df_model = df_model.loc[~matching].copy()

excluded_columns = {'participant', 'session', 'repetition', 'total time'}
feature_columns = [column for column in df_model.columns if column not in excluded_columns]
if len(df_model) != 19769 or len(feature_columns) != 46:
    raise ValueError(f'Unexpected corrected dataset: {df_model.shape}; features={len(feature_columns)}')
if feature_columns != [column for column in df_raw.columns if column not in excluded_columns]:
    raise ValueError('Feature order differs from raw CSV column order.')

df_session_1 = df_model.loc[df_model['session'].eq(1)].copy()
df_session_1['_raw_row_id'] = df_session_1.index
if len(df_session_1) != 9876 or df_session_1['participant'].nunique() != 99:
    raise ValueError('Unexpected corrected Session 1 count or class coverage.')
fit_rows = df_session_1.loc[df_session_1['repetition'].le(FIT_MAX_REPETITION)].copy()
embargo_rows = df_session_1.loc[
    df_session_1['repetition'].between(FIT_MAX_REPETITION + 1, VALIDATION_MIN_REPETITION - 1)
].copy()
validation_rows = df_session_1.loc[df_session_1['repetition'].ge(VALIDATION_MIN_REPETITION)].copy()
if (len(fit_rows), len(embargo_rows), len(validation_rows)) != (6928, 891, 2057):
    raise ValueError('Chronological attempt counts differ from the audited sequence design.')
if VALIDATION_MIN_REPETITION - FIT_MAX_REPETITION - 1 < WINDOW_LENGTH - 1:
    raise ValueError('Boundary embargo is shorter than window length minus one.')
if set(fit_rows['participant']) != set(validation_rows['participant']):
    raise ValueError('Fit and validation participants are not identical.')

fit_participant_order = sorted(fit_rows['participant'].unique())
print('Corrected Session 1 rows/features/classes:', len(df_session_1), len(feature_columns), len(fit_participant_order))
print('Fit / embargo / validation attempt rows:', len(fit_rows), len(embargo_rows), len(validation_rows))
print('Session 2 was excluded from the modeling dataframe.')

In [ ]:
def sha256_window_ids(windows):
    payload = '\n'.join(','.join(map(str, row_ids)) for row_ids in windows).encode('ascii')
    return hashlib.sha256(payload).hexdigest()

def get_ordered_window_ids(partition_rows):
    windows = []
    ordered = partition_rows.sort_values(['participant', 'repetition'])
    for participant, group in ordered.groupby('participant', sort=True):
        row_ids = group['_raw_row_id'].tolist()
        repetitions = group['repetition'].astype(int).tolist()
        run_start = 0
        for position in range(1, len(group) + 1):
            at_run_end = position == len(group) or repetitions[position] != repetitions[position - 1] + 1
            if not at_run_end:
                continue
            run_ids = row_ids[run_start:position]
            windows.extend(
                tuple(run_ids[start:start + WINDOW_LENGTH])
                for start in range(len(run_ids) - WINDOW_LENGTH + 1)
            )
            run_start = position
    return windows

fit_window_row_ids = get_ordered_window_ids(fit_rows)
validation_window_row_ids = get_ordered_window_ids(validation_rows)
fit_window_fingerprint = sha256_window_ids(fit_window_row_ids)
validation_window_fingerprint = sha256_window_ids(validation_window_row_ids)
if (len(fit_window_row_ids), len(validation_window_row_ids)) != (6021, 1166):
    raise ValueError('Window counts differ from the audited LSTM/GRU window design.')
if fit_window_fingerprint != EXPECTED_FIT_WINDOW_IDS_SHA256:
    raise ValueError('Fit window raw-row IDs do not match audited sequence design.')
if validation_window_fingerprint != EXPECTED_VALIDATION_WINDOW_IDS_SHA256:
    raise ValueError('Validation window raw-row IDs do not match audited sequence design.')
fit_window_raw_ids = {row_id for window in fit_window_row_ids for row_id in window}
validation_window_raw_ids = {row_id for window in validation_window_row_ids for row_id in window}
if fit_window_raw_ids.intersection(validation_window_raw_ids):
    raise AssertionError('A raw attempt is shared between fit and validation windows.')

participant_by_id = df_session_1.set_index('_raw_row_id')['participant'].to_dict()
repetition_by_id = df_session_1.set_index('_raw_row_id')['repetition'].astype(int).to_dict()
for windows in (fit_window_row_ids, validation_window_row_ids):
    for row_ids in windows:
        if len({participant_by_id[row_id] for row_id in row_ids}) != 1:
            raise AssertionError('A window crosses participant boundaries.')
        repetitions = [repetition_by_id[row_id] for row_id in row_ids]
        if any(right != left + 1 for left, right in zip(repetitions, repetitions[1:])):
            raise AssertionError('A window crosses a missing repetition.')

print('Fit/validation window counts:', len(fit_window_row_ids), len(validation_window_row_ids))
print('Fit/validation ID SHA256:', fit_window_fingerprint, validation_window_fingerprint)
print('Shared raw attempt IDs:', len(fit_window_raw_ids.intersection(validation_window_raw_ids)))

In [ ]:
run_records = []
history_records = []
metric_names = ['accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1']

def classification_metrics(y_true, y_pred):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0,
    )
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': macro_f1,
    }

for model_name in ['mlp_single_attempt', 'lstm_10_attempt', 'gru_10_attempt']:
    for seed in SEEDS:
        run_started = time.perf_counter()
        random.seed(seed)
        np.random.seed(seed)
        tf.keras.utils.set_random_seed(seed)

        run_scaler = StandardScaler()
        X_fit_rows = fit_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
        X_validation_rows = validation_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
        X_fit_scaled = run_scaler.fit_transform(X_fit_rows).astype(np.float32)
        X_validation_scaled = run_scaler.transform(X_validation_rows).astype(np.float32)
        run_encoder = LabelEncoder().fit(fit_rows['participant'])
        if list(run_encoder.classes_) != fit_participant_order:
            raise AssertionError('Run-specific label encoder class order changed.')
        y_fit_attempts = run_encoder.transform(fit_rows['participant'])
        y_validation_attempts = run_encoder.transform(validation_rows['participant'])
        if int(run_scaler.n_samples_seen_) != len(fit_rows):
            raise AssertionError('Scaler includes data outside fit attempt rows.')

        if model_name == 'mlp_single_attempt':
            estimator = MLPClassifier(
                hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
                activation='relu',
                solver='adam',
                batch_size=256,
                learning_rate_init=MLP_LEARNING_RATE,
                alpha=MLP_ALPHA,
                max_iter=MLP_MAX_ITER,
                early_stopping=True,
                validation_fraction=MLP_VALIDATION_FRACTION,
                n_iter_no_change=10,
                random_state=seed,
            )
            estimator.fit(X_fit_scaled, y_fit_attempts)
            y_pred = estimator.predict(X_validation_scaled)
            run_metrics = classification_metrics(y_validation_attempts, y_pred)
            loss_curve = list(estimator.loss_curve_)
            internal_scores = list(getattr(estimator, 'validation_scores_', []))
            for epoch_index, loss_value in enumerate(loss_curve):
                history_records.append({
                    'model': model_name,
                    'seed': seed,
                    'epoch': epoch_index + 1,
                    'fit_loss': loss_value,
                    'fit_accuracy': np.nan,
                    'validation_loss': np.nan,
                    'validation_accuracy': internal_scores[epoch_index] if epoch_index < len(internal_scores) else np.nan,
                    'validation_accuracy_scope': 'MLP internal early-stopping subset of fit partition',
                })
            epochs_run = int(estimator.n_iter_)
            best_epoch = int(estimator.best_iteration_) if hasattr(estimator, 'best_iteration_') else epochs_run
            early_stopping_note = 'sklearn MLPClassifier early_stopping; internal validation_fraction=0.15'
            del estimator
        else:
            scaled_by_row = dict(zip(fit_rows['_raw_row_id'].tolist(), X_fit_scaled))
            scaled_by_row.update(zip(validation_rows['_raw_row_id'].tolist(), X_validation_scaled))
            X_fit_windows = np.asarray([
                np.stack([scaled_by_row[row_id] for row_id in row_ids]) for row_ids in fit_window_row_ids
            ], dtype=np.float32)
            X_validation_windows = np.asarray([
                np.stack([scaled_by_row[row_id] for row_id in row_ids]) for row_ids in validation_window_row_ids
            ], dtype=np.float32)
            y_fit_windows = np.asarray([
                run_encoder.transform([participant_by_id[row_ids[0]]])[0]
                for row_ids in fit_window_row_ids
            ], dtype=np.int64)
            y_validation_windows = np.asarray([
                run_encoder.transform([participant_by_id[row_ids[0]]])[0]
                for row_ids in validation_window_row_ids
            ], dtype=np.int64)
            if X_fit_windows.shape != (6021, WINDOW_LENGTH, len(feature_columns)):
                raise AssertionError('Fit window tensor shape changed.')
            if X_validation_windows.shape != (1166, WINDOW_LENGTH, len(feature_columns)):
                raise AssertionError('Validation window tensor shape changed.')
            if len(np.unique(y_fit_windows)) != 99 or len(np.unique(y_validation_windows)) != 99:
                raise AssertionError('All 99 classes must occur in each window partition.')

            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(seed)
            recurrent_layer = (
                tf.keras.layers.LSTM(RECURRENT_UNITS)
                if model_name == 'lstm_10_attempt'
                else tf.keras.layers.GRU(RECURRENT_UNITS)
            )
            neural_model = tf.keras.Sequential([
                tf.keras.layers.Input(shape=(WINDOW_LENGTH, len(feature_columns))),
                recurrent_layer,
                tf.keras.layers.Dropout(DROPOUT_RATE),
                tf.keras.layers.Dense(len(run_encoder.classes_), activation='softmax'),
            ])
            neural_model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'],
            )
            callback = tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True,
            )
            fit_history = neural_model.fit(
                X_fit_windows,
                y_fit_windows,
                validation_data=(X_validation_windows, y_validation_windows),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[callback],
                verbose=0,
            )
            y_pred = neural_model.predict(X_validation_windows, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)
            run_metrics = classification_metrics(y_validation_windows, y_pred)
            epochs_run = len(fit_history.history['loss'])
            best_epoch = int(np.argmin(fit_history.history['val_loss']) + 1)
            early_stopping_note = f'val_loss; patience={EARLY_STOPPING_PATIENCE}; restore_best_weights=True'
            for epoch_index in range(epochs_run):
                history_records.append({
                    'model': model_name,
                    'seed': seed,
                    'epoch': epoch_index + 1,
                    'fit_loss': fit_history.history['loss'][epoch_index],
                    'fit_accuracy': fit_history.history['accuracy'][epoch_index],
                    'validation_loss': fit_history.history['val_loss'][epoch_index],
                    'validation_accuracy': fit_history.history['val_accuracy'][epoch_index],
                    'validation_accuracy_scope': 'chronological outer validation partition',
                })
            del neural_model, fit_history, X_fit_windows, X_validation_windows
            tf.keras.backend.clear_session()

        run_records.append({
            'model': model_name,
            'seed': seed,
            'prediction_unit': 'single attempt' if model_name == 'mlp_single_attempt' else '10 consecutive attempts',
            'fit_attempt_rows': len(fit_rows),
            'validation_attempt_rows': len(validation_rows),
            'fit_windows': len(fit_window_row_ids) if model_name != 'mlp_single_attempt' else np.nan,
            'validation_windows': len(validation_window_row_ids) if model_name != 'mlp_single_attempt' else np.nan,
            'scaler_fit_rows': int(run_scaler.n_samples_seen_),
            'best_epoch_or_iteration': best_epoch,
            'epochs_or_iterations_run': epochs_run,
            'early_stopping': early_stopping_note,
            'fit_window_ids_sha256': fit_window_fingerprint if model_name != 'mlp_single_attempt' else '',
            'validation_window_ids_sha256': validation_window_fingerprint if model_name != 'mlp_single_attempt' else '',
            **run_metrics,
            'elapsed_seconds': time.perf_counter() - run_started,
        })
        print(model_name, 'seed=', seed, 'best=', best_epoch, 'metrics=',
              {key: round(run_metrics[key], 4) for key in metric_names}, flush=True)

runs = pd.DataFrame(run_records)
if len(runs) != len(SEEDS) * 3:
    raise AssertionError(f'Expected {len(SEEDS) * 3} model-seed runs, got {len(runs)}.')
if runs.duplicated(['model', 'seed']).any():
    raise AssertionError('Duplicate model/seed records found.')
print('Completed model-seed runs:', len(runs))

In [ ]:
summary_records = []
for model_name, group in runs.groupby('model', sort=False):
    for metric in metric_names:
        summary_records.append({
            'model': model_name,
            'prediction_unit': group['prediction_unit'].iloc[0],
            'metric': metric,
            'mean': group[metric].mean(),
            'std_sample': group[metric].std(ddof=1),
            'n_seeds': group['seed'].nunique(),
            'seeds': ','.join(map(str, sorted(group['seed'].unique()))),
            'partition_protocol': 'Session 1 chronological: fit <=70; embargo 71-79; validation >=80',
            'window_length': WINDOW_LENGTH if model_name != 'mlp_single_attempt' else np.nan,
        })
summary = pd.DataFrame(summary_records)

existing_lstm = pd.read_csv(LSTM_VALIDATION_PATH).iloc[0]
existing_gru = pd.read_csv(GRU_VALIDATION_PATH).iloc[0]
for artifact, expected_hashes in [
    (existing_lstm, (EXPECTED_FIT_WINDOW_IDS_SHA256, EXPECTED_VALIDATION_WINDOW_IDS_SHA256)),
    (existing_gru, (EXPECTED_FIT_WINDOW_IDS_SHA256, EXPECTED_VALIDATION_WINDOW_IDS_SHA256)),
]:
    if (artifact['fit_window_ids_sha256'] if 'fit_window_ids_sha256' in artifact.index else expected_hashes[0]) != expected_hashes[0]:
        raise ValueError('Existing sequence validation metric uses a different fit-window definition.')
    if (artifact['validation_window_ids_sha256'] if 'validation_window_ids_sha256' in artifact.index else expected_hashes[1]) != expected_hashes[1]:
        raise ValueError('Existing sequence validation metric uses a different validation-window definition.')

same_seed_differences = []
for metric in metric_names:
    pivot = runs.loc[runs['model'].isin(['lstm_10_attempt', 'gru_10_attempt'])].pivot(
        index='seed', columns='model', values=metric,
    )
    delta = pivot['gru_10_attempt'] - pivot['lstm_10_attempt']
    same_seed_differences.append({
        'comparison': 'GRU minus LSTM on identical windows',
        'metric': metric,
        'mean_paired_difference': delta.mean(),
        'std_paired_difference': delta.std(ddof=1),
        'gru_seed_wins': int((delta > 0).sum()),
        'lstm_seed_wins': int((delta < 0).sum()),
        'ties': int((delta == 0).sum()),
    })
paired_summary = pd.DataFrame(same_seed_differences)

print('Mean and sample standard deviation across seeds:')
print(summary.to_string(index=False))
print('\nPaired same-seed GRU minus LSTM differences:')
print(paired_summary.to_string(index=False))

output_paths = [RUNS_OUTPUT_PATH, SUMMARY_OUTPUT_PATH, HISTORY_OUTPUT_PATH]
collisions = [path for path in output_paths if path.exists()]
if collisions:
    raise FileExistsError(f'Refusing to overwrite existing study outputs: {collisions}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
runs.to_csv(RUNS_OUTPUT_PATH, index=False)
summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
pd.DataFrame(history_records).to_csv(HISTORY_OUTPUT_PATH, index=False)
print('\nSaved per-seed metrics:', RUNS_OUTPUT_PATH)
print('Saved mean/std summary:', SUMMARY_OUTPUT_PATH)
print('Saved training histories:', HISTORY_OUTPUT_PATH)
print('No model artifacts were written; Session 2 was not accessed or evaluated.')